In [2]:
import sys
sys.path.append('../')

In [3]:
%load_ext autoreload
%autoreload 2

In [4]:
import numpy as np
import pandas as pd

import nltk
import wtpsplit

from src.config.paths import DATA_PATH, SUPREME_PATH, IQ2_PATH, TENNIS_PATH

# Segmentation Tools

In [ ]:
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\abi\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [ ]:
nltk.sent_tokenize('yoooo? what. the. actual. fuck. Dawg; like bro... im so done; This is a different sentence btw -- like')

['yoooo?',
 'what.',
 'the.',
 'actual.',
 'fuck.',
 'Dawg; like bro... im so done; This is a different sentence btw -- like']

wow this is ASSS

In [4]:
sat = wtpsplit.SaT('sat-3l-sm')

Loading weights:   0%|          | 0/55 [00:00<?, ?it/s]

In [5]:
sat.split('yooo lowkey idk if this works ahhah anyways kys')

['yooo lowkey idk if this works ', 'ahhah anyways kys']

# Dataset Preprocessing

## Standardizing

In [ ]:
sets = {'sup': SUPREME_PATH, 'iq2': IQ2_PATH, 'ten': TENNIS_PATH}
dfs = {}

for k,v in sets.items():
    dfs[k] = pd.read_json(
        v.joinpath('utterances.jsonl'),
        lines=True
    )

In [ ]:
for i in ['iq2', 'ten']:
    dfs[i] = dfs[i].rename(columns={
        'root' : 'conversation_id',
        'user': 'speaker'
    })

In [ ]:
dfs['ten']['conversation_id'] = dfs['ten']['conversation_id'].apply(lambda x: x.split('_')[0])

In [ ]:
for k,v in dfs.items():
    dfs[k] = v[['id', 'conversation_id', 'speaker', 'text']]

## Segmenting IQ2

In [ ]:
len(dfs['iq2'])

26562

In [ ]:
rows = []

n = 0
N = len(dfs['iq2'])
for _, row in dfs['iq2'].iterrows():
    sentences = sat.split(row.text)
    
    for i, sentence in enumerate(sentences):
        new_row = row.copy()
        new_row.id = f'{row.id}_{i}'
        new_row.text = sentence
        rows.append(new_row)

    n += 1
    print(f'\r[{n}/{N}]', end='')

new_iq2_df = pd.DataFrame(rows)

[26562/26562]

In [ ]:
new_iq2_df

,id,conversation_id,speaker,text
0,0_0,0,Bob Costas,…
0,0_1,0,Bob Costas,And now I’d like to introduce Robert Rosenkran...
0,0_2,0,Bob Costas,Bob?
0,0_3,0,Bob Costas,This is Bob.
1,1_0,0,Robert Rosenkranz,Well thank you very much.
...,...,...,...,...
26561,26561_65,26330,John Donvan,That's up 16 percent.
26561,26561_66,26330,John Donvan,And 10 percent were undecided.
26561,26561_67,26330,John Donvan,The side against the motion the two-party syst...
26561,26561_68,26330,John Donvan,Our congratulations to them.


## Merging Integrity

In [ ]:
a = set(dfs['ten']['conversation_id'].unique())
b = set(dfs['sup']['conversation_id'].unique())
c = set(new_iq2_df['conversation_id'].unique())

In [ ]:
len(a.intersection(b)), len(a.intersection(c)), len(b.intersection(c))

(0, 0, 34)

In [ ]:
new_iq2_df['conversation_id'] = new_iq2_df['conversation_id'].apply(lambda x: f'iq2_{x}')

## Dataset Ratio

In [ ]:
len(dfs['sup']), len(dfs['ten']), len(new_iq2_df)

(1700789, 163948, 129117)

we dont want the supreme court to be overrepresentated, probably best to use a 12-20k sample subset instead

In [ ]:
df = dfs['sup']
sampled = df['conversation_id'].drop_duplicates().sample(frac=0.05)

df_sampled = df[df['conversation_id'].isin(sampled)]

In [ ]:
df_sampled

,id,conversation_id,speaker,text
1976,13066__0_000,13066,charles_e_ford,"-- of this Court, the Dennis case, the one -- ..."
1977,13066__0_001,13066,j__hugo_l_black,What judge did that?
1978,13066__0_002,13066,charles_e_ford,"That was Judge Kirkland, Your Honor.\nThen, af..."
1979,13066__0_003,13066,j__earl_warren,"I think, Mr. Ford, your time has --"
1980,13066__0_004,13066,charles_e_ford,May I make one other
...,...,...,...,...
1699205,24959__3_170,24959,j__john_g_roberts_jr,Yes.
1699206,24959__3_171,24959,joseph_r_palmore,The Department of Labor has the legal authorit...
1699207,24959__3_172,24959,j__john_g_roberts_jr,"Thank you, counsel. Three minutes, Mr. Stris."
1699208,24959__4_000,24959,peter_k_stris,Thank you.\nThree brief points.\nThe first two...


dude, i just saw huge monologues in this one as well, probably good to get a < 100k subset and then break it down

## Segmenting Supreme Court Subset

In [ ]:
rows = []

n = 0
k = 0
N = len(df_sampled)
for _, row in df_sampled.iterrows():
    sentences = sat.split(row.text)
    
    for i, sentence in enumerate(sentences):
        new_row = row.copy()
        new_row.id = f'{row.id}_{i}'
        new_row.text = sentence
        rows.append(new_row)

    k += len(sentences)
    n += 1
    print(f'\r[{n}/{N}] {k}', end='')

new_sup_df = pd.DataFrame(rows)

[86887/86887] 212475

## Finally Merging the Datasets

In [ ]:
new_ten_df = dfs['ten']
new_ten_df['root'] = 'ten'

new_ten_df

,id,conversation_id,speaker,text,root
0,1681_0.q,1681,REPORTER,I think this is your biggest success right now...,ten
1,1681_0.a,1681,Kei Nishikori,Yeah.,ten
2,1681_1.q,1681,REPORTER,How would you describe it? Is it fantastic for...,ten
3,1681_1.a,1681,Kei Nishikori,"Yeah, I'm pretty happy, but it was -- I wasn't...",ten
4,1681_2.q,1681,REPORTER,Do you know why he has retired?,ten
...,...,...,...,...,...
163943,755_9.a,755,Kim Clijsters,"Yeah, no.",ten
163944,755_10.q,755,REPORTER,It's working?,ten
163945,755_10.a,755,Kim Clijsters,Yeah. Feel good.,ten
163946,755_11.q,755,REPORTER,So when something's not going right with your ...,ten


In [ ]:
new_iq2_df['root'] = 'iq2'
new_sup_df['root'] = 'sup'

In [ ]:
df = pd.concat([new_ten_df, new_iq2_df, new_sup_df])

In [ ]:
df.to_csv(DATA_PATH.joinpath('dia-merged.csv'), index=False)

## Ermmm Maybe We Need to Segment ALL of them

In [6]:
df = pd.read_csv(DATA_PATH.joinpath('dia-merged.csv'), dtype=str)

In [8]:
rows = []

n = 0
breaked_on = None
N = len(df)
for _, row in df.iterrows():
    if row.root == 'ten':
        if not breaked_on:
            sentences = sat.split(row.text)

            for i, sentence in enumerate(sentences):
                new_row = row.copy()
                new_row.id = f'{row.id}_{i}'
                new_row.text = sentence
                rows.append(new_row)

            if len(rows) > 250_000:
                breaked_on = row.conversation_id
    else:
        rows.append(row)
    n += 1
    print(f'\r[{n}/{N}] {len(rows)}', end='')

df = pd.DataFrame(rows)

[505540/505540] 591594

In [ ]:
# sampled = df['conversation_id'].isin(df[df['root'] == 'ten']['conversation_id'].drop_duplicates().sample(frac=0.3))
# non_ten = df['root'] != 'ten'

# df[sampled | non_ten]

In [17]:
(df['conversation_id'] == breaked_on).sum()

np.int64(107)

In [21]:
df = df[df['conversation_id'] != breaked_on]

In [22]:
df

,id,conversation_id,speaker,text,root
0,1681_0.q_0,1681,REPORTER,I think this is your biggest success right now...,ten
1,1681_0.a_0,1681,Kei Nishikori,Yeah.,ten
2,1681_1.q_0,1681,REPORTER,How would you describe it?,ten
2,1681_1.q_1,1681,REPORTER,Is it fantastic for you?,ten
3,1681_1.a_0,1681,Kei Nishikori,"Yeah, I'm pretty happy, but it was --",ten
...,...,...,...,...,...
505535,24959__4_000_44,24959,peter_k_stris,"Again, this shouldn't drive standing, but if i...",sup
505536,24959__4_000_45,24959,peter_k_stris,And in situations of catastrophe like AIG and ...,sup
505537,24959__4_000_46,24959,peter_k_stris,We ask that you reverse.,sup
505538,24959__4_001_0,24959,j__john_g_roberts_jr,"Thank you, counsel.",sup


In [24]:
df.to_parquet(DATA_PATH.joinpath('dia-merged2.parquet'))